In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip install sentence-transformers --quiet

import pandas as pd
import numpy as np
import torch
import ast
import os
import json
from sentence_transformers import SentenceTransformer, InputExample
from torch.utils.data import DataLoader
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

print(f"GPU disponibil: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"Memorie GPU: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

GPU disponibil: True
Device: Tesla T4
Memorie GPU: 14.6 GB


In [3]:
INPUT_PATH  = '/kaggle/input/datasets/catalinalupu/movies-with-review-summaries/movies_with_review_summaries.csv'
OUTPUT_PATH = '/kaggle/working/'

df = pd.read_csv(INPUT_PATH)
print(f"Filme incarcate: {len(df):,}")
print(f"Coloane: {df.columns.tolist()}")

df['overview']        = df['overview'].fillna('').astype(str).str.strip()
df['tagline']         = df['tagline'].fillna('').astype(str).str.strip()
df['review_summary']  = df['review_summary'].fillna('').astype(str).str.strip()
df['review_summary']  = df['review_summary'].replace('nan', '')

has_review = (df['review_summary'] != '').sum()
print(f"\nFilme cu review_summary: {has_review:,} ({has_review/len(df)*100:.1f}%)")
print(f"Filme fara review_summary: {len(df) - has_review:,}")

Filme incarcate: 40,197
Coloane: ['movie_id', 'title', 'overview', 'input_text', 'genres', 'keywords', 'director', 'release_date', 'runtime', 'popularity', 'tagline', 'review_texts', 'certification', 'cast', 'has_review', 'overview_summary', 'input_review', 'review_summary']

Filme cu review_summary: 11,644 (29.0%)
Filme fara review_summary: 28,553


In [4]:
def extract_cast_names(cast_raw, max_actors=5):
    if not isinstance(cast_raw, str) or not cast_raw.strip():
        return ''
    try:
        actors = ast.literal_eval(cast_raw)
        if isinstance(actors, list):
            names = [
                a['name'] for a in actors[:max_actors]
                if isinstance(a, dict) and a.get('name')
            ]
            return ', '.join(names)
    except Exception:
        pass
    return ''


def extract_keywords(kw_raw, max_kw=15):
    if not isinstance(kw_raw, str) or not kw_raw.strip():
        return ''
    try:
        parsed = ast.literal_eval(kw_raw)
        if isinstance(parsed, list):
            return ', '.join(str(k) for k in parsed[:max_kw])
    except Exception:
        pass
    words = kw_raw.split(',')
    return ', '.join(w.strip() for w in words[:max_kw])

def build_doc_text(row):
    parts = []

    if pd.notna(row.get('title')) and str(row['title']).strip():
        parts.append(str(row['title']).strip() + '.')

    if pd.notna(row.get('overview')) and str(row['overview']).strip():
        parts.append(str(row['overview']).strip())

    review = str(row.get('review_summary', '')).strip()
    if review and review != 'nan':
        parts.append('Reviews: ' + review + '.')

    if pd.notna(row.get('genres')) and str(row['genres']).strip():
        parts.append('Genres: ' + str(row['genres']).strip() + '.')

    kw = extract_keywords(row.get('keywords', ''))
    if kw:
        parts.append('Keywords: ' + kw + '.')

    cast = extract_cast_names(row.get('cast', ''))
    if cast:
        parts.append('Cast: ' + cast + '.')

    return ' '.join(parts)


df['doc_text'] = df.apply(build_doc_text, axis=1)

mask = (df['tagline'] != '') & (df['doc_text'] != '')
df_valid = df[mask].copy().reset_index(drop=True)

print(f"Filme cu tagline si doc_text: {len(df_valid):,}")
print(f"Filme eliminate: {len(df) - len(df_valid):,}")

sample_with    = df_valid[df_valid['review_summary'] != ''].iloc[0]
sample_without = df_valid[df_valid['review_summary'] == ''].iloc[0]

print(f"\n--- Film CU review: '{sample_with['title']}' ---")
print(f"doc_text: {sample_with['doc_text']}")
print(f"\n--- Film FARA review: '{sample_without['title']}' ---")
print(f"doc_text: {sample_without['doc_text']}")


Filme cu tagline si doc_text: 40,197
Filme eliminate: 0

--- Film CU review: 'Toy Story' ---
doc_text: Toy Story. Led by Woody, Andys toys live happily in his room until Andys birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andys heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences. Reviews: Tom Hanks leads a strong cast. Tim Allen is great, too, as Buzz Lightyear. Don Rickles (Mr. Potato Head), Wallace Shawn (Rex) and John Ratzenberger (Hamm) also bring fun.. Genres: ['Family', 'Comedy', 'Animation', 'Adventure']. Keywords: rescue, friendship, mission, jealousy, villain, bullying, elementary school, rivalry, anthropomorphism, friends, computer animation, buddy, walkie talkie, toy car, boy next door. Cast: Tom Hanks, Tim Allen, Don Rickles, Jim Varney, Wallace Shawn.

--- Film FARA review: 'Grumpier Old Men' ---
doc_text: Grumpier Old Men. A family wedding 

In [5]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained('sentence-transformers/all-mpnet-base-v2')

sample_docs = df_valid['doc_text'].sample(1000, random_state=42)
lengths = [len(tok(t, truncation=False)['input_ids']) for t in sample_docs]

docs_with_review    = df_valid[df_valid['review_summary'] != '']['doc_text'].sample(
    min(500, (df_valid['review_summary'] != '').sum()), random_state=42)
docs_without_review = df_valid[df_valid['review_summary'] == '']['doc_text'].sample(
    min(500, (df_valid['review_summary'] == '').sum()), random_state=42)

len_with    = [len(tok(t, truncation=False)['input_ids']) for t in docs_with_review]
len_without = [len(tok(t, truncation=False)['input_ids']) for t in docs_without_review]

print(f"doc_text FARA review — Medie: {np.mean(len_without):.1f} | Max: {max(len_without)} | >512: {sum(l>512 for l in len_without)}/{len(len_without)}")
print(f"doc_text CU review   — Medie: {np.mean(len_with):.1f} | Max: {max(len_with)} | >512: {sum(l>512 for l in len_with)}/{len(len_with)}")
print(f"\nGlobal — Medie: {np.mean(lengths):.1f} | 99%: {np.percentile(lengths, 99):.1f} | >512: {sum(l>512 for l in lengths)}/{len(lengths)}")

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

doc_text FARA review — Medie: 115.0 | Max: 281 | >512: 0/500
doc_text CU review   — Medie: 176.7 | Max: 320 | >512: 0/500

Global — Medie: 132.6 | 99%: 248.0 | >512: 0/1000


In [6]:
train_df, eval_df = train_test_split(df_valid, test_size=0.1, random_state=42)
print(f"Training: {len(train_df):,} filme")
print(f"Eval:     {len(eval_df):,} filme")

train_examples = [
    InputExample(texts=[row['tagline'], row['doc_text']])
    for _, row in train_df.iterrows()
    if len(row['tagline'].split()) >= 3
]

train_with_review = sum(
    1 for _, row in train_df.iterrows()
    if str(row.get('review_summary', '')).strip() not in ('', 'nan')
)

print(f"\nExemple de antrenare: {len(train_examples):,}")
print(f"  - cu review in doc_text: {train_with_review:,} ({train_with_review/len(train_df)*100:.1f}%)")
print(f"  - fara review in doc_text: {len(train_df)-train_with_review:,}")

print(f"\nExemplu pereche (cu review):")
for ex in train_examples:
    if 'Reviews:' in ex.texts[1]:
        print(f"  Query: {ex.texts[0]}")
        print(f"  Doc:   {ex.texts[1][:300]}...")
        break

Training: 36,177 filme
Eval:     4,020 filme

Exemple de antrenare: 35,788
  - cu review in doc_text: 10,438 (28.9%)
  - fara review in doc_text: 25,739

Exemplu pereche (cu review):
  Query: The prophecy chose him to lead the last battle
  Doc:   The Veil. Set in a wartorn land where tribal factions live in fear of annihilation, the film tells the story of a deadly warrior leading a destructive war campaign. When he is betrayed by his own and left for dead, he is healed by a mysterious princess and taken in by a hidden tribe that believes he...


In [7]:
MODEL_NAME   = 'all-mpnet-base-v2'
OUTPUT_MODEL = OUTPUT_PATH + 'sbert_v3'
device       = 'cuda' if torch.cuda.is_available() else 'cpu'

sbert = SentenceTransformer(MODEL_NAME, device=device)
print(f"Model {MODEL_NAME} incarcat pe {device}")
print(f"Dimensiune embedding: {sbert.encode(['test']).shape[1]}")

total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
free_mem  = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1024**3
print(f"Memorie GPU: {total_mem:.1f} GB total, {free_mem:.1f} GB libera")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model all-mpnet-base-v2 incarcat pe cuda
Dimensiune embedding: 768
Memorie GPU: 14.6 GB total, 14.1 GB libera


In [8]:
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from torch.amp import autocast, GradScaler
from tqdm.auto import tqdm

torch.cuda.empty_cache()
import gc; gc.collect()

if isinstance(sbert, torch.nn.DataParallel):
    sbert = sbert.module

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
sbert  = sbert.to(device)

_tok = sbert.tokenizer

def tokenize(texts):
    return _tok(texts, padding=True, truncation=True, max_length=256, return_tensors='pt')

def get_embeddings(encoded):
    encoded = {k: v.to(device) for k, v in encoded.items()}
    out = sbert(encoded)
    return out['sentence_embedding']

def mnr_loss(emb_a, emb_p, scale=20.0):
    scores = torch.mm(emb_a, emb_p.T) * scale
    labels = torch.arange(len(emb_a), device=device)
    return F.cross_entropy(scores, labels)

def simple_collate(examples):
    return [e.texts[0] for e in examples], [e.texts[1] for e in examples]

BATCH_SIZE  = 32
ACCUM_STEPS = 4   
EPOCHS      = 3

train_dl     = DataLoader(train_examples, batch_size=BATCH_SIZE, shuffle=True, collate_fn=simple_collate)
total_steps  = (len(train_dl) // ACCUM_STEPS) * EPOCHS
warmup_steps = int(0.1 * total_steps)

optimizer = AdamW(sbert.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler    = GradScaler('cuda')

print(f"Antrenare: {len(train_examples):,} exemple")
print(f"  batch={BATCH_SIZE} | negatives/query={BATCH_SIZE-1} | accum={ACCUM_STEPS} | epoci={EPOCHS}")

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

for epoch in range(EPOCHS):
    sbert.train()
    total_loss = 0.0
    accum_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(enumerate(train_dl), total=len(train_dl), desc=f"Epoch {epoch+1}/{EPOCHS}")
    for step, (anchors, positives) in pbar:
        enc_a = tokenize(anchors)
        enc_p = tokenize(positives)

        with autocast('cuda'):
            emb_a = get_embeddings(enc_a)
            emb_p = get_embeddings(enc_p)
            loss  = mnr_loss(emb_a, emb_p) / ACCUM_STEPS

        scaler.scale(loss).backward()
        accum_loss += loss.item()

        if (step + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(sbert.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            total_loss += accum_loss
            pbar.set_postfix({'loss': f'{accum_loss:.4f}'})
            accum_loss = 0.0

    steps_done = len(train_dl) // ACCUM_STEPS
    print(f"Epoch {epoch+1}/{EPOCHS} — Loss mediu: {total_loss / steps_done:.4f}")

    ckpt_path = OUTPUT_PATH + f'sbert_v3_epoch{epoch+1}'
    sbert.save(ckpt_path)
    print(f"  Checkpoint salvat: {ckpt_path}")

sbert.save(OUTPUT_MODEL)
print(f"\nFine-tuning complet! Model final: {OUTPUT_MODEL}")


Antrenare: 35,788 exemple
  batch=32 | negatives/query=31 | accum=4 | epoci=3


Epoch 1/3:   0%|          | 0/1119 [00:00<?, ?it/s]

Epoch 1/3 — Loss mediu: 1.4602


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Checkpoint salvat: /kaggle/working/sbert_v3_epoch1


...

Epoch 3/3 — Loss mediu: 0.9714


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  Checkpoint salvat: /kaggle/working/sbert_v3_epoch3


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Fine-tuning complet! Model final: /kaggle/working/sbert_v3


In [11]:
sbert_base = SentenceTransformer(MODEL_NAME,                               device=device) 
sbert_v3   = SentenceTransformer(OUTPUT_MODEL,                             device=device)  

queries = df_valid['tagline'].tolist()
docs    = df_valid['doc_text'].tolist()
BATCH_EMB = 256

print("Generare embeddings baseline...")
q_emb_base = sbert_base.encode(queries, batch_size=BATCH_EMB, show_progress_bar=True, convert_to_numpy=True)
d_emb_base = sbert_base.encode(docs,    batch_size=BATCH_EMB, show_progress_bar=True, convert_to_numpy=True)
print("\nGenerare embeddings v3 (fine-tuned, cu review in doc)...")
q_emb_v3 = sbert_v3.encode(queries, batch_size=BATCH_EMB, show_progress_bar=True, convert_to_numpy=True)
d_emb_v3 = sbert_v3.encode(docs,    batch_size=BATCH_EMB, show_progress_bar=True, convert_to_numpy=True)

np.save(OUTPUT_PATH + 'q_emb_base.npy', q_emb_base)
np.save(OUTPUT_PATH + 'd_emb_base.npy', d_emb_base)
np.save(OUTPUT_PATH + 'q_emb_v3.npy',  q_emb_v3)
np.save(OUTPUT_PATH + 'd_emb_v3.npy',  d_emb_v3)
print("\nEmbeddings salvate!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Generare embeddings baseline...


Batches:   0%|          | 0/158 [00:00<?, ?it/s]

Batches:   0%|          | 0/158 [00:00<?, ?it/s]


Generare embeddings v3 (fine-tuned, cu review in doc)...


Batches:   0%|          | 0/158 [00:00<?, ?it/s]

Batches:   0%|          | 0/158 [00:00<?, ?it/s]


Embeddings salvate!


In [12]:
def parse_genres(genres_raw):
    if not isinstance(genres_raw, str) or not genres_raw.strip():
        return []
    try:
        parsed = ast.literal_eval(genres_raw)
        if isinstance(parsed, list):
            return [str(g).lower().strip() for g in parsed if g]
    except Exception:
        pass
    return [g.strip().lower() for g in genres_raw.split(',') if g.strip()]

df_valid['genres_list'] = df_valid['genres'].apply(parse_genres)

genre_to_indices = {}
for i, genres in enumerate(df_valid['genres_list']):
    for g in genres:
        genre_to_indices.setdefault(g, []).append(i)

print(f"Genuri unice: {len(genre_to_indices)}")

Genuri unice: 19


In [14]:
def evaluate_full_pool(query_emb, doc_emb, k_values=[1, 3, 5, 10, 20], batch_size=512):
    N = len(query_emb)
    hits = {k: 0 for k in k_values}
    max_k = max(k_values)

    for start in range(0, N, batch_size):
        end    = min(start + batch_size, N)
        scores = cosine_similarity(query_emb[start:end], doc_emb)
        top_k  = np.argpartition(scores, -max_k, axis=1)[:, -max_k:]

        for i, global_idx in enumerate(range(start, end)):
            top_sorted = top_k[i][np.argsort(scores[i][top_k[i]])[::-1]]
            for k in k_values:
                if global_idx in top_sorted[:k]:
                    hits[k] += 1

        if start % 5000 == 0:
            print(f"  {start}/{N}...")

    return {k: hits[k] / N for k in k_values}


def evaluate_genre_filtered(query_emb, doc_emb, k_values=[1, 3, 5, 10, 20],
                             sample_n=3000, min_pool=30):
    np.random.seed(42)
    indices = np.random.choice(len(query_emb), min(sample_n, len(query_emb)), replace=False)
    hits = {k: 0 for k in k_values}
    pool_sizes = []

    for idx in indices:
        pool = set()
        for g in df_valid['genres_list'].iloc[idx]:
            pool.update(genre_to_indices.get(g, []))
        pool.add(idx)

        if len(pool) < min_pool:
            pool = set(range(len(df_valid)))

        pool_arr  = np.array(sorted(pool))
        pool_sizes.append(len(pool_arr))

        q      = query_emb[idx:idx+1]
        d_pool = doc_emb[pool_arr]
        scores = cosine_similarity(q, d_pool)[0]

        local_idx = int(np.where(pool_arr == idx)[0][0])
        ranked    = np.argsort(scores)[::-1]
        rank      = int(np.where(ranked == local_idx)[0][0]) + 1

        for k in k_values:
            if rank <= k:
                hits[k] += 1

    n = len(indices)
    print(f"  Pool mediu per query: {np.mean(pool_sizes):.0f} filme")
    return {k: hits[k] / n for k in k_values}

In [15]:
results = {}

print("FULL POOL (toate filmele)")

print("\nBaseline (fara fine-tuning):")
results['base_full'] = evaluate_full_pool(q_emb_base, d_emb_base)
print(f"  {results['base_full']}")

print("\nV3: fine-tuned, cu review + batch=32:")
results['v3_full'] = evaluate_full_pool(q_emb_v3, d_emb_v3)
print(f"  {results['v3_full']}")

print("\nGENRE-FILTERED POOL")

print("\nBaseline:")
results['base_genre'] = evaluate_genre_filtered(q_emb_base, d_emb_base)
print(f"  {results['base_genre']}")

print("\nV3: fine-tuned, cu review + batch=32:")
results['v3_genre'] = evaluate_genre_filtered(q_emb_v3, d_emb_v3)
print(f"  {results['v3_genre']}")

with open(OUTPUT_PATH + 'results_v3.json', 'w') as f:
    json.dump(results, f, indent=2)
print("\nRezultate salvate!")


FULL POOL (toate filmele)

Baseline (fara fine-tuning):
  0/40197...
  {1: 0.06535313580615469, 3: 0.09776849018583476, 5: 0.11475980794586661, 10: 0.14145334228922557, 20: 0.17240092544219718}

V3: fine-tuned, cu review + batch=32:
  0/40197...
  {1: 0.10172400925442197, 3: 0.15814613030823196, 5: 0.18896932606910963, 10: 0.23790332611886458, 20: 0.2976092743239545}

GENRE-FILTERED POOL

Baseline:
  Pool mediu per query: 15835 filme
  {1: 0.084, 3: 0.128, 5: 0.14733333333333334, 10: 0.17833333333333334, 20: 0.213}

V3: fine-tuned, cu review + batch=32:
  Pool mediu per query: 15835 filme
  {1: 0.11466666666666667, 3: 0.16733333333333333, 5: 0.19733333333333333, 10: 0.25166666666666665, 20: 0.30433333333333334}

Rezultate salvate!


In [16]:
print(f"{'Model':<50} {'Hit@1':>7} {'Hit@3':>7} {'Hit@5':>7} {'Hit@10':>7} {'Hit@20':>7}")

labels = [
    ('base_full',  'Baseline     | full pool'),
    ('v3_full',    'V3 fine-tune | full pool  (cu review, batch=32)'),
    ('base_genre', 'Baseline     | genre pool'),
    ('v3_genre',   'V3 fine-tune | genre pool (cu review, batch=32)'),
]

for key, label in labels:
    r = results[key]
    print(f"{label:<50} {r[1]:>7.3f} {r[3]:>7.3f} {r[5]:>7.3f} {r[10]:>7.3f} {r[20]:>7.3f}")

print(f"\n{'[Referinta V2 full pool]':<50} {'0.100':>7} {'0.155':>7} {'0.187':>7} {'0.236':>7} {'0.293':>7}")

print()
gain_1  = (results['v3_full'][1]  - results['base_full'][1])  / results['base_full'][1]  * 100
gain_10 = (results['v3_full'][10] - results['base_full'][10]) / results['base_full'][10] * 100
print(f"V3 vs Baseline — Hit@1: +{gain_1:.1f}%, Hit@10: +{gain_10:.1f}%")

v2_hit1 = 0.100
if results['v3_full'][1] > v2_hit1:
    print(f"V3 vs V2       — Hit@1: +{(results['v3_full'][1]-v2_hit1)/v2_hit1*100:.1f}% (imbunatatire fata de varianta2)")
else:
    print(f"V3 vs V2       — Hit@1: {(results['v3_full'][1]-v2_hit1)/v2_hit1*100:.1f}% (sub varianta2)")


Model                                                Hit@1   Hit@3   Hit@5  Hit@10  Hit@20
Baseline     | full pool                             0.065   0.098   0.115   0.141   0.172
V3 fine-tune | full pool  (cu review, batch=32)      0.102   0.158   0.189   0.238   0.298
Baseline     | genre pool                            0.084   0.128   0.147   0.178   0.213
V3 fine-tune | genre pool (cu review, batch=32)      0.115   0.167   0.197   0.252   0.304

[Referinta V2 full pool]                             0.100   0.155   0.187   0.236   0.293

V3 vs Baseline — Hit@1: +55.7%, Hit@10: +68.2%
V3 vs V2       — Hit@1: +1.7% (imbunatatire fata de varianta2)


In [18]:
def recommend(query: str, top_k: int = 5, model='v3'):
    models = {'base': (sbert_base, d_emb_base),
              'v3':   (sbert_v3,   d_emb_v3)}
    m, d = models[model]

    q_emb  = m.encode([query], convert_to_numpy=True)
    scores = cosine_similarity(q_emb, d)[0]
    top_idx = np.argsort(scores)[::-1][:top_k]

    has_rev_icon = lambda idx: '✓' if df_valid.iloc[idx]['review_summary'] not in ('', 'nan') else ' '

    print(f"\nQuery [{model}]: \"{query}\"")
    print(f"{'Rang':<5} {'Rev':>4} {'Titlu':<40} {'Gen':<28} {'Scor':>6}")
    print('-' * 90)
    for rank, idx in enumerate(top_idx, 1):
        row    = df_valid.iloc[idx]
        genres = str(row['genres'])[:26]
        print(f"{rank:<5} {has_rev_icon(idx):>4} {row['title']:<40} {genres:<28} {scores[idx]:>6.3f}")

print("Legenda: Rev ✓ = film are review_summary in doc_text")

queries_demo = [
    "animated toys that come to life and go on adventures",
    "a superhero saves the world from an alien invasion",
    "romantic comedy set in New York with funny misunderstandings",
    "psychological thriller about dreams and reality",
]

for q in queries_demo:
    recommend(q, top_k=5, model='v3')

Legenda: Rev ✓ = film are review_summary in doc_text

Query [v3]: "animated toys that come to life and go on adventures"
Rang   Rev Titlu                                    Gen                            Scor
------------------------------------------------------------------------------------------
1          Raggedy Ann & Andy: A Musical Adventure! ['Animation', 'Adventure',    0.565
2          Patch Town                               ['Music', 'Adventure', 'Co    0.560
3          Toopy and Binoo The Movie                ['Animation', 'Adventure',    0.546
4          The Christmas Toy                        ['Family', 'Comedy', 'Fant    0.543
5          The Indian in the Cupboard               ['Adventure', 'Family', 'F    0.532

Query [v3]: "a superhero saves the world from an alien invasion"
Rang   Rev Titlu                                    Gen                            Scor
------------------------------------------------------------------------------------------
1          Shin

In [29]:
from sentence_transformers import CrossEncoder

CE_MODEL = 'cross-encoder/ms-marco-MiniLM-L-6-v2'
cross_encoder = CrossEncoder(CE_MODEL, device='cuda' if torch.cuda.is_available() else 'cpu')
print(f"Cross-encoder incarcat: {CE_MODEL}")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Cross-encoder incarcat: cross-encoder/ms-marco-MiniLM-L-6-v2


In [31]:
def evaluate_with_crossencoder(query_emb, doc_emb, k_values=[1, 3, 5, 10, 20],
                                candidates=100, sample_n=3000):
    np.random.seed(42)
    indices = np.random.choice(len(query_emb), min(sample_n, len(query_emb)), replace=False)
    hits_ce = {k: 0 for k in k_values}
    hits_bi = {k: 0 for k in k_values}  

    docs_list = df_valid['doc_text'].tolist()

    for step, idx in enumerate(indices):
        sbert_scores = cosine_similarity(query_emb[idx:idx+1], doc_emb)[0]
        top_cand_idx = np.argsort(sbert_scores)[::-1][:candidates]

        for k in k_values:
            if idx in top_cand_idx[:k]:
                hits_bi[k] += 1

        query_text = df_valid['tagline'].iloc[idx]
        pairs      = [[query_text, docs_list[i]] for i in top_cand_idx]
        ce_scores  = cross_encoder.predict(pairs, show_progress_bar=False)

        reranked_local = np.argsort(ce_scores)[::-1]
        reranked_global = top_cand_idx[reranked_local]

        for k in k_values:
            if idx in reranked_global[:k]:
                hits_ce[k] += 1

        if (step + 1) % 200 == 0:
            print(f"  {step+1}/{len(indices)} | "
                  f"Hit@1 bi={hits_bi[1]/(step+1):.3f} ce={hits_ce[1]/(step+1):.3f} | "
                  f"Hit@10 bi={hits_bi[10]/(step+1):.3f} ce={hits_ce[10]/(step+1):.3f}")

    n = len(indices)
    return {k: hits_bi[k] / n for k in k_values}, {k: hits_ce[k] / n for k in k_values}


print("Evaluare Cross-Encoder pe sample de 3000 filme...")
print("(pentru evaluare completa pe 40k schimba sample_n=len(q_emb_v3) — dureaza ~2-3h)\n")

bi_scores, ce_scores_eval = evaluate_with_crossencoder(
    q_emb_v3, d_emb_v3,
    candidates=100,
    sample_n=3000
)

print(f"\n{'Model':<45} {'Hit@1':>7} {'Hit@3':>7} {'Hit@5':>7} {'Hit@10':>7} {'Hit@20':>7}")
for label, r in [('V3 bi-encoder (fara re-rank)', bi_scores),
                 ('V3 + Cross-Encoder re-rank (top100)', ce_scores_eval)]:
    print(f"{label:<45} {r[1]:>7.3f} {r[3]:>7.3f} {r[5]:>7.3f} {r[10]:>7.3f} {r[20]:>7.3f}")

gain_1  = (ce_scores_eval[1]  - bi_scores[1])  / bi_scores[1]  * 100
gain_10 = (ce_scores_eval[10] - bi_scores[10]) / bi_scores[10] * 100
print(f"\nCastig cross-encoder — Hit@1: +{gain_1:.1f}%, Hit@10: +{gain_10:.1f}%")


Evaluare Cross-Encoder pe sample de 3000 filme...
(pentru evaluare completa pe 40k schimba sample_n=len(q_emb_v3) — dureaza ~2-3h)

  200/3000 | Hit@1 bi=0.135 ce=0.120 | Hit@10 bi=0.205 ce=0.240
  400/3000 | Hit@1 bi=0.102 ce=0.115 | Hit@10 bi=0.212 ce=0.230
  600/3000 | Hit@1 bi=0.098 ce=0.112 | Hit@10 bi=0.217 ce=0.223
  800/3000 | Hit@1 bi=0.102 ce=0.125 | Hit@10 bi=0.219 ce=0.233
  1000/3000 | Hit@1 bi=0.101 ce=0.121 | Hit@10 bi=0.216 ce=0.233
  1200/3000 | Hit@1 bi=0.103 ce=0.126 | Hit@10 bi=0.220 ce=0.232
  1400/3000 | Hit@1 bi=0.103 ce=0.126 | Hit@10 bi=0.222 ce=0.226
  1600/3000 | Hit@1 bi=0.107 ce=0.131 | Hit@10 bi=0.227 ce=0.231
  1800/3000 | Hit@1 bi=0.105 ce=0.131 | Hit@10 bi=0.225 ce=0.229
  2000/3000 | Hit@1 bi=0.105 ce=0.132 | Hit@10 bi=0.227 ce=0.232
  2200/3000 | Hit@1 bi=0.105 ce=0.134 | Hit@10 bi=0.226 ce=0.233
  2400/3000 | Hit@1 bi=0.104 ce=0.132 | Hit@10 bi=0.225 ce=0.233
  2600/3000 | Hit@1 bi=0.103 ce=0.132 | Hit@10 bi=0.224 ce=0.233
  2800/3000 | Hit@1 bi=0.10